Packages installation

In [1]:
!pip install haversine

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from sklearn.metrics import matthews_corrcoef
from transformers import BertForSequenceClassification
import torch
import numpy as np
import pickle
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
import argparse
from haversine import haversine, Unit
import h3
import os
from tqdm import tqdm


In [ ]:


# This file test the first version of the model: classification with context


# PRETRAINED_MODEL_NAME = "/home/daril/scratch/data/trajcbert/models/model_saved_parallel_version_full_bs_32_20_epochs_with_context"
# DATALOADER_DIR = "/home/daril/trajcbert/savings/test_dataloader_833383.pt"


In [ ]:



# # --pretrained_model_name $PRETRAINED_MODEL_NAME \
# # --dataloader_dir $DATALOADER_DIR


# # recovery the arguments
# parser = argparse.ArgumentParser()
# parser.add_argument(
#     "--pretrained_model_name",
#     default="/home/daril/scratch/data/trajcbert/models/model_saved_parallel_version_full_bs_32_20_epochs_with_context",
#     type=str,
#     help="The pretrained model name",
# )
# parser.add_argument(
#     "--dataloader_dir",
#     default="/home/daril/trajcbert/savings/test_dataloader_833383.pt",
#     type=str,
#     help="The dataloader directory",
# )

# args = parser.parse_args()

# PRETRAINED_MODEL_NAME = args.pretrained_model_name
# DATALOADER_DIR = args.dataloader_dir

In [3]:

PRETRAINED_MODEL_NAME = '/home/daril_kw/data/model_saved_parallel_version_full_multinode'
TOKENIZER_DIR = '/home/daril_kw/data/savings_for_parallel_computing/tokenizer_final_opti_full'
# DATALOADER_DIR = "/home/daril_kw/data/savings_for_parallel_computing/test_dataloader_full.pt"
DATALOADER_DIR = "/home/daril_kw/data/savings_for_60_rows/test_dataloader_60.pt"


DIR_TARGETS = "/home/daril_kw/data/savings_for_parallel_computing/targets_full_opti.pkl"


In [4]:
!nvidia-smi

Sun Feb 16 09:44:34 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti      On | 00000000:04:00.0 Off |                  N/A |
| 29%   25C    P8               20W / 250W|    697MiB / 11264MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [5]:


# device = torch.device("cpu")
# use the GPU 1 
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# load the prediction_dataloader
prediction_dataloader = torch.load(DATALOADER_DIR)

# we load the model
model = BertForSequenceClassification.from_pretrained(PRETRAINED_MODEL_NAME)
model.to(device)
print("we evaluate")
model.eval()

# Tracking variables
predictions, true_labels, list_inputs_test = [], [], []

# losses
losses = 0
print("We predict")
# Predict

/home/daril_kw/.venv/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


we evaluate
We predict


In [6]:
with open(DIR_TARGETS, 'rb') as f:
    targets = pickle.load(f)

In [7]:



targets_dict={}
for i in range(len(targets)):
    if targets[i] not in targets_dict:
        targets_dict[targets[i]]=len(targets_dict)

targets_input=[targets_dict[targets[i]] for i in range(len(targets))]


In [8]:
# Construction de l'inverse du dictionnaire
targets_dict_inv = {v: k for k, v in targets_dict.items()}

In [9]:
def compute_absolute_distance(prediction, true_label):
    """
    This function convert the predition and the true label into h3 tokens and then convert the h3 tokens into lat, long
    And finally compute the distance between the two points
    """

    # convert the prediction and the true label into h3 tokens
    prediction_h3 = targets_dict_inv[prediction]
    true_label_h3 = targets_dict_inv[true_label]

    # convert the h3 tokens into lat, long
    prediction_lat, prediction_long = h3.h3_to_geo(prediction_h3)
    true_label_lat, true_label_long = h3.h3_to_geo(true_label_h3)

    # compute the distance between the two points
    distance = haversine((prediction_lat, prediction_long), (true_label_lat, true_label_long))

    return distance

In [14]:
# Test of the function compute_absolute_distance
print(compute_absolute_distance(0, 1))

3.355761774166987


In [10]:

for batch in tqdm(prediction_dataloader):
    # Add batch to GPU
    batch = tuple(t.to(device) for t in batch)

    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels = batch

    # move to device
    b_input_ids = b_input_ids.to(device)
    b_input_mask = b_input_mask.to(device)
    b_labels = b_labels.to(device)

    # Telling the model not to compute or store gradients, saving memory and
    # speeding up prediction
    with torch.no_grad():
        # Forward pass, calculate logit predictions
        outputs = model(
            b_input_ids, token_type_ids=None, attention_mask=b_input_mask
        )

    logits = outputs[0]
    losses += outputs[0].mean().item()

    # Move logits and labels to CPU
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to("cpu").numpy()

    # Store predictions and true labels
    predictions.append(logits)
    true_labels.append(label_ids)

    # Store the inputs

    list_inputs_test.append(b_input_ids.tolist())
    #free the memory
    del b_input_ids
    del b_input_mask
    del b_labels
    del outputs
    del logits
    del label_ids
    torch.cuda.empty_cache()
    

print("DONE.")


  0%|                                                                                                          | 0/1 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.05s/it]

DONE.


In [21]:

matthews_set = []
mae_set = []

# Evaluate each test batch using Matthew's correlation coefficient
print("Calculating Matthews Corr. Coef. for each batch...")

pred_label = []
# compute the loss

# For each input batch...
for i in tqdm(range(len(true_labels)), desc="Calculating Matthews Corr. Coef., MAE and accuracy"):
    # The predictions for this batch are a 2-column ndarray (one column for "0"
    # and one column for "1"). Pick the label with the highest value and turn this
    # in to a list of 0s and 1s.
    pred_labels_i = np.argmax(predictions[i], axis=1).flatten()
    print(f" pred_labels_i: {pred_labels_i}") # it content a list of the predition for each batch
    print(f" true_labels : {true_labels}, lenh: {len(true_labels)}")
    print(f" true_labels[i]: {true_labels[i]}")
    
    pred_label.append(pred_labels_i)
    # Calculate and store the coef for this batch.
    matthews = matthews_corrcoef(true_labels[i], pred_labels_i)
    matthews_set.append(matthews)

    # compute the distance between the prediction and the true label
    # distances = compute_absolute_distance(tuple(pred_labels_i), tuple(true_labels)[i])
    for j in range(len(pred_labels_i)):
        distances = compute_absolute_distance(pred_labels_i[j], true_labels[i][j])
        mae_set.append(abs(distances))

    # free the memory
    torch.cuda.empty_cache()

# Combine the predictions for each batch into a single list of 0s and 1s.
flat_predictions = [item for sublist in predictions for item in sublist]
flat_predictions = np.argmax(flat_predictions, axis=1).flatten()

# Combine the correct labels for each batch into a single list.
flat_true_labels = [item for sublist in true_labels for item in sublist]

# Combine the inputs for each batch into a single list.
flat_list_inputs_test = [item for sublist in list_inputs_test for item in sublist]


# Compute MAE
mae = np.mean(mae_set)

print("MAE: %.3f" % mae)
# Calculate the MCC
mcc = matthews_corrcoef(flat_true_labels, flat_predictions)

print("MCC: %.3f" % mcc)

# compute the accuracy
accuracy = (flat_true_labels == flat_predictions).mean()
print("accuracy: %.3f" % accuracy)

# print the loss
# print("loss: %.3f" % (losses / len(true_labels)))

# save flat_list_inputs_test


Calculating Matthews Corr. Coef. for each batch...


Calculating Matthews Corr. Coef., MAE and accuracy: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 152.27it/s]

 pred_labels_i: [2093  584 2735 4255 6134 1239  320 1178 6422 5812  761  560]
 true_labels : [array([39, 34, 18, 14, 12, 48, 10,  0,  9, 40, 42, 36])], lenh: 1
 true_labels[i]: [39 34 18 14 12 48 10  0  9 40 42 36]
MAE: 4.317
MCC: 0.000
accuracy: 0.000
